# BenchFlow Scene Patterns

Six evaluation patterns — multi-turn, multi-round, multi-scene — expressed as Scene configs.

| Term | Definition |
|------|------------|
| **Turn** | One prompt in one ACP session |
| **Round** | One A→B exchange between roles |
| **Scene** | Interaction region with roles + turns |
| **Trial** | Sequence of scenes in shared sandbox |

In [1]:
from benchflow.trial import TrialConfig, Scene, Role, Turn
from pathlib import Path

## Pattern 1: Single-Agent Baseline

In [2]:
config = TrialConfig(
    task_path=Path("tasks/regex-log"),
    scenes=[Scene.single(agent="gemini", model="gemini-3.1-flash-lite-preview")],
    environment="daytona",
)
s = config.effective_scenes[0]
print(f"{len(config.effective_scenes)} scene, {len(s.roles)} role, {len(s.turns)} turn")

1 scene, 1 role, 1 turn


**Result:** reward=0.0, 3 tool calls. Agent wrote a regex but missed edge cases.

## Pattern 2: Single-Agent Multi-Turn
Same agent, two prompts. Self-review variant.

In [3]:
config = TrialConfig(
    task_path=Path("tasks/regex-log"),
    scenes=[Scene(
        name="self-review",
        roles=[Role("agent", "gemini", "gemini-3.1-flash-lite-preview")],
        turns=[
            Turn("agent"),
            Turn("agent", "Review your solution. Check edge cases and fix issues."),
        ],
    )],
    environment="daytona",
)
s = config.effective_scenes[0]
print(f"{len(config.effective_scenes)} scene, {len(s.roles)} role, {len(s.turns)} turns")

1 scene, 1 role, 2 turns


**Result:** reward=1.0, 12 tool calls. Self-review caught the regex bug.

## Pattern 3: Multi-Round Code Review
Two roles — coder and reviewer — take turns.

In [4]:
config = TrialConfig(
    task_path=Path("tasks/regex-log"),
    scenes=[Scene(
        name="code-review",
        roles=[
            Role("coder", "gemini", "gemini-3.1-flash-lite-preview"),
            Role("reviewer", "gemini", "gemini-3.1-flash-lite-preview"),
        ],
        turns=[
            Turn("coder"),
            Turn("reviewer", "Review the code. Write feedback to /app/.outbox/coder.json"),
            Turn("coder", "Read feedback and fix issues."),
        ],
    )],
    environment="daytona",
)
s = config.effective_scenes[0]
print(f"{len(config.effective_scenes)} scene, {len(s.roles)} roles, {len(s.turns)} turns")

1 scene, 2 roles, 3 turns


**Result:** reward=0.0, 16 tool calls.

**At scale (267 trials on TB2):** baseline 9.0% → reviewer 19.4% win rate.

## Pattern 4: Interactive User Simulation
A "user" role reveals info gradually. Agent responds.

In [5]:
config = TrialConfig(
    task_path=Path("tasks/regex-log"),
    scenes=[Scene(
        name="interactive",
        roles=[
            Role("user", "gemini", "gemini-3.1-flash-lite-preview"),
            Role("agent", "gemini", "gemini-3.1-flash-lite-preview"),
        ],
        turns=[
            Turn("user", "Give the agent a vague version of the task..."),
            Turn("agent"),
        ],
    )],
    environment="daytona",
)
s = config.effective_scenes[0]
print(f"{len(config.effective_scenes)} scene, {len(s.roles)} roles, {len(s.turns)} turns")

1 scene, 2 roles, 2 turns


**Result:** reward=0.0, 2 tool calls. Vague instruction wasn't enough.

## Pattern 5: Multi-Scene BYOS
Two scenes — skill generation (unscored) then solve (scored).

In [6]:
config = TrialConfig(
    task_path=Path("tasks/regex-log"),
    scenes=[
        Scene(name="skill-gen",
              roles=[Role("gen", "gemini", "gemini-3.1-flash-lite-preview")],
              turns=[Turn("gen", "Write a skill document to /app/generated-skill.md")]),
        Scene(name="solve",
              roles=[Role("solver", "gemini", "gemini-3.1-flash-lite-preview")],
              turns=[Turn("solver")]),
    ],
    environment="daytona",
)
print(f"{len(config.effective_scenes)} scenes: {', '.join(f'{s.name} ({len(s.turns)} turn)' for s in config.effective_scenes)}")

2 scenes: skill-gen (1 turn), solve (1 turn)


**Result:** reward=0.0, 5 tool calls. Self-generated skills = 0pp lift (consistent with paper).

## Pattern 6: Contract Consultation
Realistic multi-round: client CTO + advisor reviewing a vendor contract.

In [7]:
config = TrialConfig(
    task_path=Path("tasks/contract-review"),
    scenes=[Scene(
        name="consultation",
        roles=[
            Role("client", "gemini", "gemini-3.1-flash-lite-preview"),
            Role("advisor", "gemini", "gemini-3.1-flash-lite-preview"),
        ],
        turns=[
            Turn("client"),   # shares situation
            Turn("advisor"),  # analyzes contract
            Turn("client", "Clarify your top 2 priorities."),
            Turn("advisor", "Give final redline recommendations."),
        ],
    )],
    environment="daytona",
)
s = config.effective_scenes[0]
print(f"{len(config.effective_scenes)} scene, {len(s.roles)} roles, {len(s.turns)} turns ({len(s.turns)//2} rounds)")

1 scene, 2 roles, 4 turns (2 rounds)


Each pattern is a TrialConfig — same API, same lifecycle, same verifier. The only difference is the Scene configuration.